In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# 1. Kloning repositori proyekmu
!git clone https://github.com/temu-isamrofni/gccp-small-llm-reranking.git
%cd gccp-small-llm-reranking

In [ ]:
!pip install -r requirements.txt
!pip install transformers accelerate bitsandbytes scikit-learn scipy datasets

In [ ]:
!python scripts/prepare_data.py

In [ ]:
# Sel Notebook untuk memuat model dengan kuantisasi 4-bit agar pas di GPU T4 16GB
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"
# model_id = "Qwen2.5-1.5B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# Catatan: Jika menggunakan Llama-3, pastikan kamu sudah memasukkan HuggingFace Token di rahasia Kaggle (Kaggle Secrets)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model sukses dimuat dalam mode kuantisasi 4-bit!")

In [ ]:
!python scripts/run_bm25.py --config configs/scifact.yaml

In [ ]:
from src.config import load_config
from src.data import load_corpus, load_queries
from src.pointwise_reranker import rerank_pointwise
from src.run_io import load_run, save_run

# 1. Muat konfigurasi eksperimen SciFact
config = load_config("configs/scifact.yaml")
queries = load_queries(config.dataset.queries_path)
corpus = load_corpus(config.dataset.corpus_path)

# 2. Ambil kandidat dokumen hasil pencarian BM25 dari Sel 1
bm25_run_path = config.outputs.run_dir / f"{config.experiment_name}_bm25.txt"
candidates = load_run(bm25_run_path)

# 3. Tentukan device akselerator GPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Memulai Pointwise Reranking dengan Qwen2.5-3B-Instruct...")
pointwise_run = rerank_pointwise(
    queries=queries,
    corpus=corpus,
    candidates=candidates,
    top_k=config.reranking.top_k,
    batch_size=4,  # Mengonsumsi VRAM jauh lebih sedikit dan aman dari OOM
    model=model,          # Objek model di memorimu
    tokenizer=tokenizer,  # Objek tokenizer di memorimu
    device=device
)

# 4. Simpan hasil pemeringkatan pointwise
pointwise_output_path = config.outputs.run_dir / f"{config.experiment_name}_pointwise.txt"
save_run(pointwise_output_path, pointwise_run, tag="POINTWISE")
print(f"Pointwise selesai! Hasil disimpan ke: {pointwise_output_path}")

In [ ]:
from src.gccp_reranker import rerank_gccp
from src.pagc import aggregate_pagc

print("Memulai GCCP Reranking dengan Qwen2.5-7B (Spectral Anchor)...")
gccp_run = rerank_gccp(
    queries=queries,
    corpus=corpus,
    candidates=candidates,
    top_k=config.reranking.top_k,
    anchor_top_k=config.reranking.anchor_top_k,
    model=model,
    tokenizer=tokenizer,
    device=device,
    anchor_weight=0.25 # Bobot kontribusi global context (f_c)
)

print("Menggabungkan hasil (Aggregation) ke dalam skema PAGC...")
pagc_run = aggregate_pagc(
    pointwise_run=pointwise_run,
    gccp_run=gccp_run,
    top_k=config.reranking.top_k,
)

# Simpan kedua berkas keluaran eksperimen utama
gccp_output_path = config.outputs.run_dir / f"{config.experiment_name}_gccp.txt"
pagc_output_path = config.outputs.run_dir / f"{config.experiment_name}_pagc.txt"

save_run(gccp_output_path, gccp_run, tag="GCCP")
save_run(pagc_output_path, pagc_run, tag="PAGC")

print(f"Eksperimen Selesai!\nGCCP Run -> {gccp_output_path}\nPAGC Run -> {pagc_output_path}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.config import load_config
from src.data import load_corpus, load_queries
from src.gccp_reranker import rerank_gccp
from src.pagc import aggregate_pagc
from src.run_io import load_run, save_run

# 1. Muat konfigurasi dasar eksperimen SciFact
config = load_config("configs/scifact.yaml")
queries = load_queries(config.dataset.queries_path)
corpus = load_corpus(config.dataset.corpus_path)

# 2. LANGKAH KRUSIAL: Muat hasil komputasi Pointwise yang sudah ada (Menghemat 2.8 Jam!)
pointwise_run_path = "/kaggle/input/datasets/edpootis/qwenqwen2-5-3b-instruct-bm25-pointwise/scifact_pointwise.txt"
pointwise_run = load_run(pointwise_run_path)
print(f"👉 Sukses memuat {len(pointwise_run)} hasil Pointwise dari disk. Fase ini resmi dilewati!")

# Muat kandidat awal BM25 sebagai input fasa kontrastif GCCP
bm25_run_path = "/kaggle/input/datasets/edpootis/qwenqwen2-5-3b-instruct-bm25-pointwise/scifact_bm25.txt"
candidates = load_run(bm25_run_path)

# 3. Muat model SLM yang lebih ringan untuk mengantisipasi panjangnya prompt GCCP
# Pilihan terbaik: Qwen/Qwen2.5-3B-Instruct
model_id = "Qwen/Qwen2.5-3B-Instruct" 

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print(f"Memuat model ramah VRAM: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 4. Jalankan HANYA GCCP Reranking (Gunakan versi Batching teroptimasi sebelumnya)
print("Memulai GCCP Reranking ...")
gccp_run = rerank_gccp(
    queries=queries,
    corpus=corpus,
    candidates=candidates,
    top_k=config.reranking.top_k,
    anchor_top_k=config.reranking.anchor_top_k,
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_size=4  # Batasan aman anti-OOM untuk fasa contrastive global context
)

# 5. Agregasi PAGC Akhir
print("Menggabungkan skor akhir menggunakan skema PAGC...")
pagc_run = aggregate_pagc(
    pointwise_run=pointwise_run,
    gccp_run=gccp_run,
    top_k=config.reranking.top_k,
)

# 6. Simpan seluruh hasil akhir eksperimen riil
gccp_output_path = config.outputs.run_dir / f"{config.experiment_name}_gccp.txt"
pagc_output_path = config.outputs.run_dir / f"{config.experiment_name}_pagc.txt"

save_run(gccp_output_path, gccp_run, tag="GCCP")
save_run(pagc_output_path, pagc_run, tag="PAGC")

print("Eksperimen selesai")

In [ ]:
!python scripts/evaluate.py --config configs/scifact.yaml --run results/runs/scifact_pagc.txt